# Assignment 2 — Exploratory Data Analysis

Starter notebook for understanding raw data, validating label definition, and planning **train / validation / test / OOT** splits.

**Sections in this notebook:**
1. Data coverage (time span & volume)
2. LMS product structure checks (loan amount, instalments, repeat borrowers)
3. Label maturity & recommended temporal split for modelling + monitoring
4. Bad-rate curve (MOB=6 label validation)
5. Financials data quality peek

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")
plt.rcParams.update({"axes.grid": False, "axes.titlesize": 16, "figure.titlesize": 18})

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

loans = pd.read_csv(DATA / "lms_loan_daily.csv", parse_dates=["loan_start_date", "snapshot_date"])
attributes = pd.read_csv(DATA / "features_attributes.csv", parse_dates=["snapshot_date"])
financials = pd.read_csv(DATA / "features_financials.csv", parse_dates=["snapshot_date"])
clickstream = pd.read_csv(DATA / "feature_clickstream.csv", parse_dates=["snapshot_date"])

print("Loans:", loans.shape)
print("Attributes:", attributes.shape)
print("Financials:", financials.shape)
print("Clickstream:", clickstream.shape)

## 1. Data coverage — time span & volume

Use this table to decide **how much history** you have for train/val/test/OOT and whether all feature sources overlap.

In [ ]:
def coverage_summary(df: pd.DataFrame, date_col: str, name: str) -> dict:
    return {
        "dataset": name,
        "rows": f"{len(df):,}",
        "customers": f"{df['Customer_ID'].nunique():,}",
        "min_date": df[date_col].min().date(),
        "max_date": df[date_col].max().date(),
        "months": df[date_col].dt.to_period("M").nunique(),
    }

# Loan-level snapshot (one row per loan per monthly LMS snapshot)
loan_apps = loans[loans["installment_num"] == 0].copy()

coverage = pd.DataFrame(
    [
        coverage_summary(loans, "snapshot_date", "lms_loan_daily (all snapshots)"),
        coverage_summary(loan_apps, "loan_start_date", "lms_loan_daily (new applications)"),
        coverage_summary(attributes, "snapshot_date", "features_attributes"),
        coverage_summary(financials, "snapshot_date", "features_financials"),
        coverage_summary(clickstream, "snapshot_date", "feature_clickstream"),
    ]
)
coverage

**Key takeaways for split design**
- **25 monthly application cohorts** (`2023-01` → `2025-01`), ~500 loans/month → **12,500 loans total**
- **1 row per customer** in attributes/financials; LMS confirms **12,500 unique customers = 12,500 loans**
- LMS daily snapshots run **`2023-01` → `2025-11`** (11 rows per loan: instalment 0–10)
- Clickstream ends **`2024-12`** and covers **8,974 customers** (not every borrower has clickstream history)
- Attributes/financials end **`2025-01`** (aligned with last application cohort)

## 2. LMS structure checks — fixed product & single loan per customer

Confirm assumptions before modelling (impacts leakage rules and customer-level vs loan-level joins).

In [ ]:
# --- Fixed loan amount? ---
unique_principals = loans.groupby("loan_id")["loan_amt"].nunique()
principal_value = loans["loan_amt"].dropna().unique().tolist()

# --- Fixed instalment schedule? ---
instalment_check = loans.groupby("loan_id").agg(
    tenure=("tenure", "first"),
    n_snapshots=("snapshot_date", "count"),
    max_installment=("installment_num", "max"),
)

due_inst = loans[loans["installment_num"] >= 1]
due_amt_values = due_inst["due_amt"].dropna().unique().tolist()
due_amt_per_loan = due_inst.groupby("loan_id")["due_amt"].nunique()

# --- One loan per customer? ---
loans_per_customer = loans.groupby("Customer_ID")["loan_id"].nunique()

product_checks = pd.DataFrame(
    {
        "check": [
            "All loans have same principal ($10,000)",
            "All loans have tenure = 10 months",
            "All loans have 11 LMS snapshots (instalment 0–10)",
            "Due amount constant at $1,000 per instalment (after origination)",
            "No customer has more than 1 loan",
            "Customers with >1 loan (count)",
        ],
        "result": [
            (unique_principals == 1).all() and principal_value == [10000],
            (instalment_check["tenure"] == 10).all(),
            (instalment_check["n_snapshots"] == 11).all() and (instalment_check["max_installment"] == 10).all(),
            (due_amt_per_loan == 1).all() and due_amt_values == [1000.0],
            (loans_per_customer == 1).all(),
            int((loans_per_customer > 1).sum()),
        ],
    }
)
product_checks

**Confirmed:** this is a **single-product** portfolio — every loan is **$10,000**, **10-month tenure**, **$1,000 monthly instalment**, and **each customer has exactly one loan**. Safe to treat `Customer_ID` ↔ `loan_id` as 1:1 for joins and to split by **`loan_start_date` cohort** without within-customer leakage.

## 3. Label maturity & train / validation / test / OOT split

Our label (Assignment 1): **default = DPD ≥ 30 at MOB 6** (`installment_num = 6`).

- Labels for a cohort are **observable ~6 calendar months** after `loan_start_date` (e.g. Jan-2023 apps → Jul-2023 snapshot).
- Split on **`loan_start_date` (application month)**, not random rows — avoids temporal leakage.
- **OOT** = most recent application cohorts reserved for **monitoring / stability** (score drift, PSI, bad-rate vs predicted).

In [ ]:
MOB_INSTALLMENT = 6

mob6 = loans[loans["installment_num"] == MOB_INSTALLMENT].copy()
# Raw LMS CSV: use overdue_amt > 0 as quick proxy; silver/A1 pipeline uses DPD >= 30
mob6["default_flag"] = (mob6["overdue_amt"] > 0).astype(int)

cohort_stats = (
    loan_apps.groupby("loan_start_date")
    .agg(n_loans=("loan_id", "count"))
    .reset_index()
)

label_stats = (
    mob6.groupby("loan_start_date")
    .agg(
        label_obs_snapshot=("snapshot_date", "first"),
        bad_rate=("default_flag", "mean"),
    )
    .reset_index()
)

cohort_stats = cohort_stats.merge(label_stats, on="loan_start_date", how="left")
cohort_stats["label_obs_snapshot"] = pd.to_datetime(cohort_stats["label_obs_snapshot"])

# Labels are only usable for modelling once MOB-6 is observable on/before the feature cutoff.
LABEL_MATURITY_AS_OF = clickstream["snapshot_date"].max()
cohort_stats["label_matured"] = cohort_stats["label_obs_snapshot"] <= LABEL_MATURITY_AS_OF
cohort_stats.loc[~cohort_stats["label_matured"], "bad_rate"] = pd.NA

cohort_stats["cumulative_loans"] = cohort_stats["n_loans"].cumsum()
cohort_stats.head(8)

In [ ]:
# Recommended temporal split (by application cohort) — adjust after reviewing bad-rate stability
SPLIT = {
    "train": ("2023-01-01", "2024-02-01"),      # 14 cohorts, ~7.0k loans — learn patterns
    "validation": ("2024-03-01", "2024-06-01"), # 4 cohorts,  ~2.0k — model / threshold selection
    "test": ("2024-07-01", "2024-10-01"),       # 4 cohorts,  ~2.0k — unbiased pre-OOT metrics
    "oot_monitoring": ("2024-11-01", "2025-01-01"),  # 3 cohorts, ~1.5k — production drift monitoring
}


def assign_split(start_date: pd.Timestamp) -> str:
    for split_name, (start, end) in SPLIT.items():
        if pd.Timestamp(start) <= start_date <= pd.Timestamp(end):
            return split_name
    return "out_of_scope"


cohort_stats["split"] = cohort_stats["loan_start_date"].apply(assign_split)

split_summary = (
    cohort_stats.groupby("split", observed=True)
    .agg(
        cohorts=("loan_start_date", "count"),
        loans=("n_loans", "sum"),
        avg_bad_rate=("bad_rate", "mean"),
        period_start=("loan_start_date", "min"),
        period_end=("loan_start_date", "max"),
    )
    .reindex(["train", "validation", "test", "oot_monitoring"])
)
split_summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Monthly application volume
axes[0].plot(
    cohort_stats["loan_start_date"],
    cohort_stats["n_loans"],
    marker="o",
    color="#4C72B0",
)
axes[0].set_title("Monthly new loans", fontsize=16)
axes[0].grid(False)
axes[0].set_ylabel("# loans")

# Bad rate at MOB 6 by cohort (matured labels only — aligns with feature cutoff)
matured = cohort_stats.dropna(subset=["bad_rate"])
last_mature_cohort = matured["loan_start_date"].max()
axes[1].plot(matured["loan_start_date"], matured["bad_rate"], marker="o", color="#4C72B0")
axes[1].axhline(matured["bad_rate"].mean(), color="gray", linestyle="--", label="overall mean (matured)")
axes[1].axvline(
    last_mature_cohort,
    color="red",
    linestyle=":",
    linewidth=1.5,
    label=f"last matured cohort ({last_mature_cohort:%Y-%m})",
)
axes[1].set_title(
    f"Default rate by application cohort (MOB=6, matured through {last_mature_cohort:%Y-%m})",
    fontsize=16,
)
axes[1].grid(False)
axes[1].set_ylabel("default rate")
axes[1].set_xlabel("loan_start_date")
axes[1].legend()
plt.tight_layout()
plt.show()

### Split design notes (for deck + `utils/ml/train.py`)

| Set | Purpose | Suggested period | ~Loans |
|-----|---------|------------------|--------|
| **Train** | Fit model | Jan-2023 → Feb-2024 | ~7,000 |
| **Validation** | Pick model / threshold | Mar-2024 → Jun-2024 | ~2,000 |
| **Test** | Report final offline AUC/F1 | Jul-2024 → Oct-2024 | ~2,000 |
| **OOT** | Airflow monitoring & stability charts | Nov-2024 → Jan-2025 | ~1,500 |

**Important constraints:**
1. Use **time-based split only** — never shuffle rows (same customer = same loan anyway).
2. Features must be **as-of application** (your A1 `g2_training_table` PIT join enforces this).
3. OOT cohorts still have matured labels in LMS (data to Nov-2025), so you can compute **actual vs predicted** bad rates in monitoring.
4. Clickstream missing for ~28% of customers — monitor whether OOT cohorts have different clickstream coverage.

In [ ]:
# Quick peek at loan timeline coverage
loans.groupby("loan_start_date")["loan_id"].nunique().head(20)

In [ ]:
# TODO: Plot bad rate by month-on-book to validate MOB=6 label cutoff
mob = loans.copy()
mob["overdue_flag"] = (mob["overdue_amt"] > 0).astype(int)
bad_rate = mob.groupby("installment_num")["overdue_flag"].mean()

plt.figure(figsize=(8, 4))
bad_rate.plot(marker="o")
plt.axvline(6, color="red", linestyle="--", label="MOB=6 (current scaffold)")
plt.title("Overdue rate by installment (proxy for default exploration)", fontsize=16)
plt.grid(False)
plt.xlabel("Installment number")
plt.ylabel("Overdue rate")
plt.legend()
plt.show()

### Analysis: overdue rate by installment (MOB)

| Installment (MOB) | Overdue rate | Δ vs prior month |
|-------------------|-------------|------------------|
| 0 (origination) | 0.0% | — |
| 1 | 5.7% | +5.7 pp |
| 2 | 11.2% | +5.5 pp |
| 3 | 17.9% | +6.7 pp |
| 4 | 23.4% | +5.5 pp |
| 5 | **28.8%** | +5.4 pp |
| 6–10 | **28.8%** | 0.0 pp (flat) |

**Headline number:** the portfolio overdue rate **plateaus at ~28.8%** from **MOB 5 onward** (MOB 6–10 are identical in this dataset). That supports using **MOB ≥ 6** as the label observation point — the bad rate has already stabilised, so we are not cutting off too early.

**What this means for ML**

1. **Label definition is stable.** Once a borrower is overdue at mid-tenure, they rarely “cure” before loan end. A default label at MOB 6 (DPD ≥ 30 in our A1 pipeline) is a **stable, business-defensible** target — not a noisy early signal.

2. **Class imbalance (~29% bad rate).** Expect roughly a 1:2.5 positive:negative ratio. Use class-balanced models and PR-AUC alongside ROC-AUC; accuracy alone will be misleading.

3. **Train / val / test / OOT split (Section 3) stays valid.** Split by **`loan_start_date` cohort**, not random rows. Because each customer has exactly one loan, there is no within-customer leakage across splits.

4. **OOT monitoring has enough signal.** With ~500 loans/month and ~29% bad rate, each monthly OOT cohort has ~140 defaults — enough to track **predicted vs actual bad rate** and score drift in the Airflow monitoring pipeline.

5. **Why not label earlier?** MOB 1–4 rates are still rising (+5–7 pp per month). Labelling before MOB 5 would capture **transient** delinquency that later stabilises, inflating false positives and making the model look better in-sample but worse in production.

**Link to Assignment 1:** our gold `label_store` uses **30 DPD @ MOB 6** — consistent with this curve and with the sample Assignment 1 decks.

In [ ]:
# TODO: Missingness / dtype audit for financials (dirty strings, outliers)
financials.head()

## 5. Feature signal — default-rate lift by decile

How much credit-risk signal does each feature carry? For each candidate driver we rank applicants into 10 deciles and measure the **observed `30dpd@mob6` default rate per decile**. A strong, monotonic spread between the best and worst decile is direct evidence the feature separates good from bad borrowers. This produces the *"worst vs best decile"* lift numbers used on the **Feature Design & Choice** slide and saves a chart to `outputs/charts/feature_decile_lift.png`.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from utils.config import CHARTS_DIR
from utils.ml.dataset import load_training_frame
sns.set_theme(style="white")
plt.rcParams.update({"axes.grid": False, "axes.titlesize": 16, "figure.titlesize": 18})

df = load_training_frame("2024-12-01")
overall = df["default_flag"].mean()

# Key credit drivers to profile (skip any that are degenerate / all-null).
features = [
    "fin_Interest_Rate", "fin_Outstanding_Debt", "fin_Num_Credit_Inquiries",
    "fin_debt_to_income", "fin_Annual_Income", "fin_credit_history_months",
]

rows, panels = [], []
for c in features:
    if c not in df.columns:
        continue
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().sum() < 100 or s.nunique() < 10:
        continue
    decile = pd.qcut(s.rank(method="first"), 10, labels=False) + 1  # 1=low .. 10=high
    by = df.groupby(decile)["default_flag"].mean()
    panels.append((c, by))
    rows.append({"feature": c, "best_decile_bad_rate": round(by.min(), 3),
                 "worst_decile_bad_rate": round(by.max(), 3),
                 "lift_x": round(by.max() / max(by.min(), 1e-6), 2)})

lift_table = pd.DataFrame(rows).sort_values("lift_x", ascending=False)

# Plot one panel per feature: bad rate across deciles vs the portfolio average.
n = len(panels); cols = 3; rws = int(np.ceil(n / cols))
fig, axes = plt.subplots(rws, cols, figsize=(5 * cols, 3.4 * rws))
for ax, (c, by) in zip(np.ravel(axes), panels):
    ax.bar(by.index.astype(int), by.values, color="#4C72B0")
    ax.axhline(overall, ls="--", color="grey")
    ax.set_title(c.replace("fin_", ""), fontsize=14)
    ax.grid(False)
    ax.set_xlabel("decile (1=low \u2192 10=high)"); ax.set_ylabel("default rate"); ax.set_ylim(0, 0.7)
for ax in np.ravel(axes)[n:]:
    ax.axis("off")
fig.suptitle(
    f"Default rate by feature decile (portfolio avg = {overall:.1%}, dashed)",
    y=1.02,
    fontsize=18,
)
plt.tight_layout()
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(CHARTS_DIR / "feature_decile_lift.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {CHARTS_DIR / 'feature_decile_lift.png'}")
lift_table